In [1]:
# VGG16 特征相似性分析
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import pdf2image
from scipy.stats import pearsonr
from scipy.spatial.distance import cosine
import os
import torch

# 设置中文字体
plt.rcParams['font.sans-serif'] = ['DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"使用设备: {device}")

使用设备: cuda


In [2]:
# 加载VGG16模型
from torchvision import models, transforms
import torch

vgg = models.vgg16(pretrained=True)
vgg.eval()
vgg = vgg.to(device)

# 图像预处理
preprocess = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

print("VGG16模型加载完成")

/home/ubuntu/.conda/envs/spike_sorting/lib/python3.11/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/ubuntu/.conda/envs/spike_sorting/lib/python3.11/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


VGG16模型加载完成


In [3]:
# 从PDF中提取图片
pdf_path = '/media/ubuntu/sda/Monkey/semantic/direct_train_monkeyF/full_reconstructions.pdf'
output_folder = '/media/ubuntu/sda/Monkey/semantic/direct_train_monkeyF/extracted_images'
os.makedirs(output_folder, exist_ok=True)

# 精确坐标
vertical_lines = [20, 1110, 1210, 2300, 2400, 3490]  # 6条竖线
horizontal_lines = [115, 1205, 1250, 2340]  # 4条横线

print("正在将PDF转换为图像...")
images = pdf2image.convert_from_path(pdf_path)
print(f"共有 {len(images)} 页")

# 提取所有图片
all_extracted_images = []

for page_idx, page_img in enumerate(images):
    page_images = []
    
    for row in range(2):
        for col in range(3):
            # 正确的索引方式
            left = vertical_lines[col * 2]
            right = vertical_lines[col * 2 + 1]
            top = horizontal_lines[row * 2]
            bottom = horizontal_lines[row * 2 + 1]
            
            crop_img = page_img.crop((left, top, right, bottom))
            page_images.append(crop_img)
    
    all_extracted_images.append(page_images)

print(f"提取完成！共 {len(all_extracted_images)} 个样本")
print(f"每样本图片数: {len(all_extracted_images[0])}")

正在将PDF转换为图像...
共有 100 页
提取完成！共 100 个样本
每样本图片数: 6


In [4]:
# VGG16多層特征提取
class VGG16FeatureExtractor(torch.nn.Module):
    def __init__(self, model):
        super().__init__()
        self.features = model.features
        self.avgpool = model.avgpool
        self.classifier = model.classifier
        
    def forward(self, x):
        features = {}
        
        # Conv1 (conv1_1 + relu + conv1_2 + relu + pool)
        x = self.features[0](x)   # conv1_1
        x = self.features[1](x)   # relu1
        x = self.features[2](x)   # conv1_2
        x = self.features[3](x)   # relu2
        x = self.features[4](x)   # pool1
        features['conv1'] = x
        
        # Conv2
        x = self.features[5](x)   # conv2_1
        x = self.features[6](x)   # relu3
        x = self.features[7](x)   # conv2_2
        x = self.features[8](x)   # relu4
        x = self.features[9](x)   # pool2
        features['conv2'] = x
        
        # Conv3
        x = self.features[10](x)  # conv3_1
        x = self.features[11](x)  # relu5
        x = self.features[12](x)  # conv3_2
        x = self.features[13](x)  # relu6
        x = self.features[14](x)  # conv3_3
        x = self.features[15](x)  # relu7
        x = self.features[16](x)  # pool3
        features['conv3'] = x
        
        # Conv4
        x = self.features[17](x)  # conv4_1
        x = self.features[18](x)  # relu8
        x = self.features[19](x)  # conv4_2
        x = self.features[20](x)  # relu9
        x = self.features[21](x)  # conv4_3
        x = self.features[22](x)  # relu10
        x = self.features[23](x)  # pool4
        features['conv4'] = x
        
        # Conv5
        x = self.features[24](x)  # conv5_1
        x = self.features[25](x)  # relu11
        x = self.features[26](x)  # conv5_2
        x = self.features[27](x)  # relu12
        x = self.features[28](x)  # conv5_3
        x = self.features[29](x)  # relu13
        x = self.features[30](x)  # pool5
        features['conv5'] = x
        
        # FC6
        x = x.view(x.size(0), -1)
        x = self.classifier[0](x)
        x = torch.relu(x)
        features['fc6'] = x
        
        # FC7
        x = self.classifier[3](x)
        x = torch.relu(x)
        features['fc7'] = x
        
        return features

vgg_extractor = VGG16FeatureExtractor(vgg)
vgg_extractor.eval()

def extract_vgg_features(img):
    img_tensor = preprocess(img).unsqueeze(0).to(device)
    with torch.no_grad():
        features = vgg_extractor(img_tensor)
    # 展平并转为numpy
    for key in features:
        features[key] = features[key].squeeze().cpu().numpy()
    return features

print("VGG16特征提取器定义完成")

VGG16特征提取器定义完成


In [5]:
# 提取所有图片的VGG16特征
print("正在提取VGG16特征...")
all_features = []

for sample_idx, sample_imgs in enumerate(all_extracted_images):
    sample_features = []
    for img in sample_imgs:
        features = extract_vgg_features(img)
        sample_features.append(features)
    all_features.append(sample_features)
    
    if (sample_idx + 1) % 20 == 0:
        print(f"已处理 {sample_idx + 1}/{len(all_extracted_images)} 个样本")

print(f"\n特征提取完成！共 {len(all_features)} 个样本")

正在提取VGG16特征...
已处理 20/100 个样本
已处理 40/100 个样本
已处理 60/100 个样本
已处理 80/100 个样本
已处理 100/100 个样本

特征提取完成！共 100 个样本


In [6]:
# 计算Pearson相关系数
def pearson_corr(vec1, vec2):
    vec1 = vec1.reshape(-1)
    vec2 = vec2.reshape(-1)
    corr, _ = pearsonr(vec1, vec2)
    return corr

layer_names = ['conv1', 'conv2', 'conv3', 'conv4', 'conv5', 'fc6', 'fc7']

# 原始图片与重建图片的相关性
print("计算原始图片与重建图片的Pearson相关系数...")
pearson_results = {layer: [] for layer in layer_names}

for sample_idx, sample_features in enumerate(all_features):
    original_features = sample_features[0]
    
    for recon_idx in range(1, 6):  # 5个重建图片
        recon_features = sample_features[recon_idx]
        for layer in layer_names:
            corr = pearson_corr(original_features[layer], recon_features[layer])
            pearson_results[layer].append(corr)

print("\n各层的Pearson相关系数 (原始vs重建):")
print("-" * 50)
for layer in layer_names:
    mean_corr = np.mean(pearson_results[layer])
    std_corr = np.std(pearson_results[layer])
    print(f"  {layer}: {mean_corr:.4f} ± {std_corr:.4f}")
print("-" * 50)

计算原始图片与重建图片的Pearson相关系数...

各层的Pearson相关系数 (原始vs重建):
--------------------------------------------------
  conv1: 0.1700 ± 0.0844
  conv2: 0.0837 ± 0.0364
  conv3: 0.0854 ± 0.0351
  conv4: 0.0801 ± 0.0426
  conv5: 0.0773 ± 0.0547
  fc6: 0.1046 ± 0.0900
  fc7: 0.1220 ± 0.1167
--------------------------------------------------


In [7]:
# Shuffle对比实验
print("\n" + "="*70)
print("Shuffle对比实验")
print("="*70)

# 收集所有原始图片和重建图片的特征
all_original_features = {layer: [] for layer in layer_names}
all_recon_features = {layer: [] for layer in layer_names}

for sample_features in all_features:
    for layer in layer_names:
        all_original_features[layer].append(sample_features[0][layer])
        for recon_idx in range(1, 6):
            all_recon_features[layer].append(sample_features[recon_idx][layer])

# 打乱计算 - 生成500个配对以匹配原始-重建配对数量
np.random.seed(42)
shuffle_results = {layer: [] for layer in layer_names}

for layer in layer_names:
    original_arr = np.array([f.reshape(-1) for f in all_original_features[layer]])  # shape: (100, feature_dim)
    recon_arr = np.array([f.reshape(-1) for f in all_recon_features[layer]])        # shape: (500, feature_dim)
    
    # 将原始图片重复5次，以匹配500个重建图片
    original_expanded = np.tile(original_arr, (5, 1))  # shape: (500, feature_dim)
    
    # 打乱重建图片
    shuffled_indices = np.random.permutation(len(recon_arr))
    recon_shuffled = recon_arr[shuffled_indices]
    
    # 计算500个shuffle配对
    for i in range(len(original_expanded)):
        corr = pearson_corr(original_expanded[i], recon_shuffled[i])
        shuffle_results[layer].append(corr)

# 打印对比结果
print("\n各层的平均Pearson相关系数对比:")
print("-" * 70)
print(f"{'Layer':<10} {'真实配对':<18} {'Shuffle后':<18} {'差异':<10}")
print("-" * 70)

comparison_results = {}
for layer in layer_names:
    real_mean = np.mean(pearson_results[layer])
    shuffle_mean = np.mean(shuffle_results[layer])
    diff = real_mean - shuffle_mean
    comparison_results[layer] = (real_mean, shuffle_mean, diff)
    print(f"{layer:<10} {real_mean:.4f} ± {np.std(pearson_results[layer]):.4f}   "
          f"{shuffle_mean:.4f} ± {np.std(shuffle_results[layer]):.4f}    {diff:+.4f}")
print("-" * 70)


Shuffle对比实验

各层的平均Pearson相关系数对比:
----------------------------------------------------------------------
Layer      真实配对               Shuffle后           差异        
----------------------------------------------------------------------
conv1      0.1700 ± 0.0844   0.1189 ± 0.0608    +0.0511
conv2      0.0837 ± 0.0364   0.0568 ± 0.0281    +0.0269
conv3      0.0854 ± 0.0351   0.0606 ± 0.0267    +0.0249
conv4      0.0801 ± 0.0426   0.0510 ± 0.0306    +0.0291
conv5      0.0773 ± 0.0547   0.0273 ± 0.0353    +0.0500
fc6        0.1046 ± 0.0900   0.0322 ± 0.0644    +0.0725
fc7        0.1220 ± 0.1167   0.0215 ± 0.0744    +0.1005
----------------------------------------------------------------------


In [8]:
# 分析：原始图片之间的相关性
print("\n" + "="*70)
print("分析：原始图片之间的Pearson相关性")
print("="*70)

original_pairwise = {layer: [] for layer in layer_names}

for i in range(len(all_features)):
    for j in range(i + 1, len(all_features)):
        for layer in layer_names:
            corr = pearson_corr(all_features[i][0][layer], all_features[j][0][layer])
            original_pairwise[layer].append(corr)

print("\n各层原始图片之间的Pearson相关系数:")
print("-" * 70)
print(f"{'Layer':<10} {'原始vs重建':<18} {'原始vs原始':<18} {'差异':<10}")
print("-" * 70)

for layer in layer_names:
    orig_vs_recon = np.mean(pearson_results[layer])
    orig_vs_orig = np.mean(original_pairwise[layer])
    diff = orig_vs_recon - orig_vs_orig
    print(f"{layer:<10} {orig_vs_recon:.4f} ± {np.std(pearson_results[layer]):.4f}   "
          f"{orig_vs_orig:.4f} ± {np.std(original_pairwise[layer]):.4f}    {diff:+.4f}")
print("-" * 70)

print("\n结论:")
print("- 原始vs重建 > 原始vs原始 → 重建图片保留了原始图片的语义特征")
print("- 差异越大，说明重建质量越好")


分析：原始图片之间的Pearson相关性

各层原始图片之间的Pearson相关系数:
----------------------------------------------------------------------
Layer      原始vs重建             原始vs原始             差异        
----------------------------------------------------------------------
conv1      0.1700 ± 0.0844   0.1089 ± 0.0592    +0.0612
conv2      0.0837 ± 0.0364   0.0508 ± 0.0267    +0.0329
conv3      0.0854 ± 0.0351   0.0578 ± 0.0249    +0.0276
conv4      0.0801 ± 0.0426   0.0437 ± 0.0264    +0.0364
conv5      0.0773 ± 0.0547   0.0232 ± 0.0353    +0.0541
fc6        0.1046 ± 0.0900   0.0259 ± 0.0583    +0.0787
fc7        0.1220 ± 0.1167   0.0188 ± 0.0749    +0.1032
----------------------------------------------------------------------

结论:
- 原始vs重建 > 原始vs原始 → 重建图片保留了原始图片的语义特征
- 差异越大，说明重建质量越好


In [9]:
from matplotlib.backends.backend_pdf import PdfPages


with PdfPages('/media/ubuntu/sda/Monkey/semantic/direct_train_monkeyF/vgg16_correlation_optimized.pdf') as pdf:
    fig, ax = plt.subplots(figsize=(6, 3.5))

    x = np.arange(len(layer_names))
    width = 0.32

    real_means = [np.mean(pearson_results[l]) for l in layer_names]
    shuffle_means = [np.mean(shuffle_results[l]) for l in layer_names]
    # Create bars with professional colors
    bars1 = ax.bar(x + width/2, real_means, width, 
                label='Original vs Reconstructed', color='#EC6F7E', alpha=1)

    bars2 = ax.bar(x - width/2, shuffle_means, width, 
                label='Shuffle (Random)', color='#5E9FD1', alpha=1)

    ax.set_xticks(x)
    ax.set_xticklabels(layer_names)
    ax.set_ylim(0, max(real_means) + 0.1)

    plt.tight_layout()
    pdf.savefig()
    plt.close()



In [10]:
n_samples = len(all_features)  # 100 samples
n_recons = 5  # 5 reconstructions per sample
n_layers = len(layer_names)  # 7 layers
n_pairs = n_samples * n_recons  # 500 pairs

data_list = []

pair_idx = 0
for sample_idx in range(n_samples):
    for recon_idx in range(1, n_recons + 1):  # 重建图片索引1-5
        for layer_idx, layer in enumerate(layer_names):
            real_corr = pearson_results[layer][pair_idx]
            shuffle_corr = shuffle_results[layer][pair_idx]
            
            data_list.append({
                'sample_idx': sample_idx,
                'recon_idx': recon_idx,
                'pair_idx': pair_idx,
                'layer': layer,
                'layer_idx': layer_idx,
                'real_correlation': real_corr,
                'shuffle_correlation': shuffle_corr,
                'correlation_gain': real_corr - shuffle_corr
            })
        pair_idx += 1

df = pd.DataFrame(data_list)

correlation_3d = np.zeros((n_pairs, n_layers, 2))

for idx, row in df.iterrows():
    pair_idx = int(row['pair_idx'])
    layer_idx = int(row['layer_idx'])
    correlation_3d[pair_idx, layer_idx, 0] = row['real_correlation']
    correlation_3d[pair_idx, layer_idx, 1] = row['shuffle_correlation']

df.to_csv('/media/ubuntu/sda/Monkey/semantic/direct_train_monkeyF/correlation_dataframe.csv', index=False)
